<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook A04: Handling Outliers</h2>
</div>

Worked solutions to the 4 exercises in
[Notebook A04: Handling Outliers](../notebooks/A04_Handling_outliers.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

The store, the open days, the seasonal profile and the residuals from the notebook.

In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")

sales = pd.read_csv(nb_config.ROSSMANN_TRAIN_PATH, parse_dates=["Date"], low_memory=False)

store = (
    sales[sales["Store"] == 1]
    .set_index("Date")
    .sort_index()
    .loc[:, ["Sales", "Customers", "Open", "Promo", "StateHoliday", "SchoolHoliday"]]
)

opened = store[store["Open"] == 1].copy()
opened["month"] = opened.index.month
opened["weekday"] = opened.index.dayofweek
opened["expected"] = opened.groupby(["month", "weekday"])["Sales"].transform("median")
opened["residual"] = opened["Sales"] - opened["expected"]


def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


low, high = iqr_bounds(opened["residual"])
baseline_flags = set(opened.index[(opened["residual"] < low) | (opened["residual"] > high)])

print(f"{len(opened)} open days")
print(f"Residual-IQR flags from the notebook: {len(baseline_flags)}")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Apply the IQR rule separately within each calendar month (group by `opened.index.month` and compute the bounds per group). How many days does it flag now, and in which months?

In [ ]:
per_month_flags = []

for month, group in opened.groupby("month"):
    month_low, month_high = iqr_bounds(group["Sales"])
    flagged = group.index[(group["Sales"] < month_low) | (group["Sales"] > month_high)]
    per_month_flags.extend(flagged)

print(f"Flagged within calendar months: {len(per_month_flags)}")
print(f"Flagged by the global IQR rule (section 4): 16")
print()
print("By month:")
print(pd.Series([timestamp.month for timestamp in per_month_flags])
      .value_counts().sort_index().to_string())

**Five days, spread across January, May, September and November — and not one in December.**

Compare that with the global rule from section 4 of the notebook, which flagged 16 days of which 14 were in
December. Grouping by calendar month has removed the December cluster entirely, because within December the
busy pre-Christmas days are no longer unusual: they are being compared against other Decembers rather than
against the year as a whole.

This is the same correction the notebook makes in section 5, arrived at by a cruder route. Bounds computed
per month absorb the yearly seasonality directly, without ever building a residual, and for a quick look at
a series that is often enough.

Where it falls short is what the per-month approach cannot absorb. It still compares Mondays with Sundays
inside each month, so the weekly cycle remains in the spread, which is why this flags five days where the
residual method flags nineteen: the bounds here are wider than they need to be. The residual approach in
section 5 removes both cycles at once and is correspondingly more sensitive.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> The `Promo` column marks promotion days, which lift sales by around 1,000 on average. Add it to the grouping (`['month', 'weekday', 'Promo']`) and recompute the residuals and flags. Does accounting for promotions remove any of the days flagged above?

In [ ]:
with_promo = opened.copy()
with_promo["expected"] = (
    with_promo.groupby(["month", "weekday", "Promo"])["Sales"].transform("median")
)
with_promo["residual"] = with_promo["Sales"] - with_promo["expected"]

promo_low, promo_high = iqr_bounds(with_promo["residual"])
promo_flags = set(
    with_promo.index[(with_promo["residual"] < promo_low) | (with_promo["residual"] > promo_high)]
)

print(f"Residual std without Promo: {opened['residual'].std():.0f}")
print(f"Residual std with Promo:    {with_promo['residual'].std():.0f}")
print()
print(f"IQR bounds without Promo: ({low:.0f}, {high:.0f})")
print(f"IQR bounds with Promo:    ({promo_low:.0f}, {promo_high:.0f})")
print()
print(f"Flagged without Promo: {len(baseline_flags)}")
print(f"Flagged with Promo:    {len(promo_flags)}")
print(f"  removed by adding Promo: {len(baseline_flags - promo_flags)}")
print(f"  newly flagged:           {len(promo_flags - baseline_flags)}")

**Yes — seven of the nineteen disappear. And thirty-four new ones appear, so the total rises from 19 to
46.**

That is not what the question leads you to expect, and the reason is worth working through.

Adding `Promo` to the grouping does exactly what it should: the residual standard deviation falls from 881
to 611, a 31% reduction. Promotions really do explain a large share of the variation, and the seven days
that stop being flagged are days that looked extreme only because a promotion was not accounted for.

But the IQR bounds are computed **from the residuals themselves**. A smaller residual spread means narrower
bounds, and narrower bounds catch more points. Better modelling has not made the data cleaner; it has made
the yardstick shorter.

So the honest answer to "does accounting for promotions help?" is: it improves the *explanation* and
changes the *question*. The 46 days flagged now are days that are unusual given the month, the weekday
**and** the promotion status — a stricter and more meaningful standard than before, and one that will
naturally identify more days.

In [ ]:
# The seven days that a promotion explains
explained = sorted(baseline_flags - promo_flags)

pd.DataFrame({
    "Sales": opened.loc[explained, "Sales"],
    "Promo": opened.loc[explained, "Promo"],
    "expected without Promo": opened.loc[explained, "expected"].round(0),
    "expected with Promo": with_promo.loc[explained, "expected"].round(0),
})

Most of the seven are promotion days whose expected value rises once promotions are modelled, closing the
gap that made them look anomalous.

If you wanted the comparison to be about the *data* rather than about the changing yardstick, you would
hold the bounds fixed — keep the original thresholds and ask how many days the better model rescues. That
is a different and equally reasonable question, and the answer would be the seven above.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-3">Exercise 3</h3>
</div>

> Re-run the ESD test with `alpha=0.20`. How many days does it flag, and how does the list compare with the days the IQR rule found?

In [ ]:
def generalized_esd(values, max_outliers=30, alpha=0.05):
    """As in the notebook: iteratively test the most extreme remaining point."""
    values = np.asarray(values, dtype=float)
    n = len(values)

    working, positions = values.copy(), np.arange(n)
    test_statistics, critical_values, candidates = [], [], []

    for i in range(1, max_outliers + 1):
        std = working.std(ddof=1)
        if std == 0:
            break
        deviations = np.abs(working - working.mean()) / std
        worst = deviations.argmax()

        test_statistics.append(deviations[worst])
        candidates.append(positions[worst])

        size = n - i + 1
        t_critical = stats.t.ppf(1 - alpha / (2 * size), size - 2)
        critical_values.append(
            (size - 1) * t_critical / np.sqrt((size - 2 + t_critical**2) * size)
        )

        working = np.delete(working, worst)
        positions = np.delete(positions, worst)

    n_outliers = 0
    for i, (statistic, critical) in enumerate(zip(test_statistics, critical_values), start=1):
        if statistic > critical:
            n_outliers = i

    return np.sort(np.array(candidates[:n_outliers], dtype=int))


rows = []
for alpha in (0.05, 0.10, 0.20, 0.50):
    positions = generalized_esd(opened["residual"].values, max_outliers=30, alpha=alpha)
    dates = opened.index[positions]
    rows.append({
        "alpha": alpha,
        "flagged": len(positions),
        "also flagged by IQR": len(set(dates) & baseline_flags),
        "dates": ", ".join(d.strftime("%Y-%m-%d") for d in dates),
    })

pd.DataFrame(rows).set_index("alpha")

**At alpha = 0.20 the test flags two days**, 24 and 31 December 2013, against one at the notebook's alpha
= 0.05. Both are in the IQR rule's list of nineteen, so ESD's findings remain a strict subset.

The striking thing is how little moves. Quadrupling alpha from 0.05 to 0.20 buys exactly one extra
detection, and even at alpha = 0.50 — a significance level nobody would defend — the count only reaches
three. Compare that with the IQR rule's nineteen at its default setting.

The reason is that these are different kinds of instrument. The IQR rule applies a fixed cut-off: anything
beyond 1.5 interquartile ranges is flagged, and on 781 days a fair number of points qualify. ESD is a
**hypothesis test with a multiple-comparison correction built in**. Its critical value already accounts for
the fact that you are testing the most extreme of hundreds of points, and that correction dominates the
choice of alpha. Loosening the significance level barely dents it.

Which behaviour you want follows from the cost of a false positive. **Screening data before modelling**
wants the generous list, because inspecting a few extra days is cheap. **Raising alerts a human must act
on** wants the strict one. And it is worth noticing that at every alpha, the first thing ESD finds is 31
December — the single most anomalous day in the series by any measure.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-4">Exercise 4</h3>
</div>

> Load the Air Quality dataset (`nb_config.AIR_QUALITY_PATH`, with `sep=';'` and `decimal=','`). Missing readings in that file are coded as `-200` rather than left empty. Find them, replace them with `NaN`, and then run the residual-based detection from section 5 on the `C6H6(GT)` column using hour-of-day as the seasonal grouping. What would have happened if you had left the -200 values in?

In [ ]:
air_quality = pd.read_csv(nb_config.AIR_QUALITY_PATH, sep=";", decimal=",")

# The file carries two trailing empty columns and some empty rows
air_quality = air_quality.dropna(axis=1, how="all").dropna(how="all")

air_quality["timestamp"] = pd.to_datetime(
    air_quality["Date"] + " " + air_quality["Time"].str.replace(".", ":", regex=False),
    format="%d/%m/%Y %H:%M:%S",
)
air_quality = air_quality.set_index("timestamp")

benzene_raw = air_quality["C6H6(GT)"]
sentinel = benzene_raw == -200

print(f"{len(benzene_raw):,} hourly readings")
print(f"Coded as -200: {int(sentinel.sum())} ({sentinel.mean():.1%})")

In [ ]:
benzene = benzene_raw.replace(-200, np.nan)

pd.DataFrame({
    "with -200 left in": benzene_raw.describe()[["mean", "std", "min", "max"]],
    "with -200 as NaN": benzene.describe()[["mean", "std", "min", "max"]],
}).round(1)

The summary statistics answer most of the question before any outlier detection runs. Treating the
sentinel as data pulls the mean from **10.1 down to 1.9** and inflates the standard deviation from **7.4 to
41.4**. Neither number describes benzene concentration; they describe a mixture of benzene concentration
and a code that means "no reading".

Now the detection itself, on the cleaned series.

In [ ]:
def residual_outliers(series, grouping):
    """Section 5's method: subtract a seasonal profile, then apply the IQR rule."""
    profile = series.groupby(grouping).transform("median")
    residual = series - profile
    lower, upper = iqr_bounds(residual)
    return (residual < lower) | (residual > upper)


clean_flags = residual_outliers(benzene, benzene.index.hour)
dirty_flags = residual_outliers(benzene_raw, benzene_raw.index.hour)

print(f"On the cleaned series:        {int(clean_flags.sum()):4d} flagged "
      f"({clean_flags.mean():.1%})")
print(f"With -200 left in the data:   {int(dirty_flags.sum()):4d} flagged "
      f"({dirty_flags.mean():.1%})")
print()
print(f"  of those, sentinel rows:    {int((dirty_flags & sentinel).sum()):4d}")
print(f"  genuine readings caught:    {int((dirty_flags & ~sentinel).sum()):4d}")
print(f"  genuine readings missed vs the clean run: "
      f"{int(clean_flags.sum()) - int((dirty_flags & ~sentinel).sum())}")

**Leaving the -200 values in fails in three ways at once**, and only the first is obvious.

**It finds the sentinel.** Of the 741 points flagged, 366 are simply the missing-data code. The detector
is working perfectly and answering a question nobody asked: it has identified that -200 is unlike the rest
of the data, which we knew.

**It misses genuine outliers.** The clean run flags 429 real readings; the contaminated run finds only 375
of them. Fifty-four genuine anomalies slip through, because 366 values sitting at -200 inflate the
interquartile range and push the bounds outward. This is the same mechanism as section 3 of the notebook,
where 161 closed days widened the 3-sigma bounds so far that the rule flagged nothing at all.

**And it corrupts everything downstream.** Any mean, standard deviation, correlation or model fitted on
this column inherits the -200s. The outlier detector at least produces a visibly strange answer; a
regression would produce a plausible-looking one.

The general rule: **find out how missing data is encoded before computing anything.** A column of
floats with no NaNs is not evidence that nothing is missing — it may only be evidence that the sentinel
value is a number. `-200`, `-999`, `9999` and `0` are all common, and each of them is silently wrong in a
different way.

---

Back to [Notebook A04](../notebooks/A04_Handling_outliers.ipynb), or on to
[Notebook A05](../notebooks/A05_Forecasting_baselines.ipynb).